# 9.10 信心和正確率對得上嗎？


某一批題模型都報信心接近0.9，結果只有一半答對。如果使用者把0.9當成「十次約九次對」，就會過度信任。另一批信心0.6而十次約六次對，則較符合信心的含義。本節把預測按信心分組，再比較每組實際答對比例，這叫可靠度分箱。

前置是[9.9高softmax機率不保證正確](https://birdhackor.github.io/tiny-perceptron-vlm/9.9.html)與[W.3的tensor平均](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。分箱是把0到1切成區間，例如兩箱是0到0.5與0.5到1。每箱平均信心取該箱所有報告數字的平均；正確率則把答對記1、答錯記0後平均。分箱不能創造真值，必須先用獨立標準判對錯。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.alignment import reliability_bins

confidence = torch.tensor([0.55, 0.65, 0.85, 0.95])
correct = torch.tensor([1, 0, 1, 0])
bins = reliability_bins(confidence, correct, bins=2)
for row in bins:
    print(
        "區間",
        row["left"],
        row["right"],
        "筆數",
        row["count"],
        "信心",
        round(row["confidence"], 2),
        "正確率",
        row["accuracy"],
    )
ece = sum(row["count"] / len(confidence) * abs(row["accuracy"] - row["confidence"]) for row in bins)
print("平均校準差", round(ece, 2))

輸入四筆信心與對應正確標記。專案工具`reliability_bins`把同一區間的筆數、平均信心、正確率整理成字典；沒有樣本的箱不會出現在結果裡。因此只輸出0.5到1.0一箱，筆數4、信心0.75、正確率0.5。這不是有兩箱就必須有兩行；我們沒有低信心樣本。

最後的式子算每箱信心與正確率差多少，取絕對值，再乘該箱筆數佔總數的比例，加總得到0.25。英文ECE是Expected Calibration Error，在這裡可讀成按樣本量加權的校準差。只有一箱，所以權重4/4=1，差即0.75−0.5。若畫橫軸信心、縱軸正確率的圖，這個點在(0.75,0.5)，低於理想的相等線，顯示過度自信。

四筆只能說明計算，不能穩定估計真實模型。箱太細時每箱樣本很少，結果會受一兩筆左右；換分箱方式也會改變ECE。若要調整溫度，應在驗證集選設定，再用未參與選擇的測試集報告，避免針對這四個結果調數字後宣稱泛化成功。

分箱是統計整理，不是把每筆信心都改成該箱正確率。四筆各自仍為0.55、0.65、0.85、0.95，平均0.75只在描述這組。如果改成四箱，前兩筆可能進0.5到0.75、後兩筆進0.75到1；每箱各兩筆、正確率都0.5，平均信心卻為0.6與0.9。箱數改變後圖點位置不同，但答對兩筆的原事實不變。看可靠度圖時要同時查箱界、筆數與原結果，不能只看一條漂亮曲線。

練習新增信心0.25且答對的一筆，兩個tensor都要各加一個值，先預測低箱新增一筆、高箱仍四筆。低箱平均信心0.25、正確率1，ECE變成(1/5)×0.75+(4/5)×0.25=0.35。執行核對，並解釋為什麼這個新樣本讓整體差變大：它在低信心處表現得比報告更好，不是出現了新錯答。
